In [1]:
from database.manager import DatabaseManager
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter
import pandas as pd

db = DatabaseManager()
sznlty = Seasonality(db)
plotter = SeasonalityPlotter()

In [2]:
result = sznlty.outright_seasonality("CO", "Z", start_year=2014, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="CO Z — outright seasonality")

In [3]:
result = sznlty.expression_seasonality("CO", "Z26- F27", start_year=2014, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="CO Z26 — seasonality")

In [4]:
def export_to_excel(result, df_enhanced, filename="seasonality_analysis.xlsx"):
    df_export = df_enhanced.copy()
    df_export.index.name = "Days to Expiry"

    # 1. Identify the current (latest) year
    current_year = max(result['series'].keys())
    
    # 2. Extract the date mapping for the current year
    # This maps 'Days to Expiry' -> 'Calendar Date'
    current_date_map = result['series'][current_year]['date']
    
    # 3. Insert it as the first column (Position 0)
    # This makes it the second column in Excel (immediately after the Index)
    df_export.insert(0, 'Current_Year_Date', current_date_map)

    # Add standard stats
    df_export['STAT_Average'] = result['average']
    df_export['STAT_Std_Dev'] = result.get('std')
    df_export['STAT_Rolling_2Sigma_Path'] = result.get('rolling_std_path')

    df_export = df_export.sort_index()

    try:
        # Convert date to string format for cleaner Excel display
        if 'Current_Year_Date' in df_export.columns:
            df_export['Current_Year_Date'] = df_export['Current_Year_Date'].dt.strftime('%Y-%m-%d')

        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df_export.to_excel(writer, sheet_name='Price Alignment')
            if result.get('warnings'):
                pd.DataFrame(result['warnings'], columns=['Warnings']).to_excel(writer, sheet_name='Metadata')
        
        print(f"✅ Data successfully exported with dates to {filename}")
    except Exception as e:
        print(f"❌ Failed to export Excel: {e}")

In [5]:
from analysis.feature_creation import FeatureCreator

# 1. Initialize FeatureCreator
fc = FeatureCreator()

# 2. Identify Anomaly Years (using the RMSE/Volatility logic)
anomaly_map = fc.get_anomaly_years(result['combined'])

# 3. Create Enhanced DataFrame with Cleaned Averages
# This adds Avg_5y_Clean, Avg_10y_Clean, and Avg_15y_Clean
df_enhanced = fc.get_cleaned_averages(result['combined'], anomaly_map)

# 4. Add Current Year Rank (1 = Highest)
df_enhanced['Current_Rank'] = fc.calculate_current_rank(result['combined'])

# 5. Calculate Statistical Metrics
# Adds UpRate, Expected, and MaxDrawdown for 3D, 6D, 10D, 15D, 21D intervals
stats_df = fc.calculate_stats(result['combined'])

# 6. Join stats with the enhanced dataframe
df_final = df_enhanced.join(stats_df)

# 7. Export to Excel
# result: for original stats and dates, df_final: for all new columns
export_to_excel(result, df_final, filename="seasonality_report_final.xlsx")

# 8. Quick Verification
print("--- Analysis Complete ---")
print(f"Anomalies Found: {anomaly_map}")
print(f"Total Columns Exported: {len(df_final.columns)}")

✅ Data successfully exported with dates to seasonality_report_final.xlsx
--- Analysis Complete ---
Anomalies Found: {'bracket_15y': [2020, 2022, 2023], 'bracket_10y': [2022, 2023], 'bracket_5y': [2022]}
Total Columns Exported: 62
